In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import numpy as np
from PIL import Image

# Force full precision to avoid numerical issues
torch.set_default_tensor_type(torch.FloatTensor)

# Try to eliminate memory issues by starting clean
torch.cuda.empty_cache()

# Print CUDA information
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"CUDA memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

# Load model with more conservative settings
model_id = "stabilityai/stable-diffusion-2-1"
try:
    # First try - full precision on CPU
    print("Loading model in full precision on CPU (safest option)...")
    pipe = StableDiffusionPipeline.from_pretrained(model_id)
    
    # Generate a tiny image with just a few steps
    print("Generating image...")
    prompt = "a bright red circle on white background"  # Simple high-contrast image
    
    # Generate with full debugging info
    with torch.no_grad():
        image = pipe(
            prompt,
            height=256,
            width=256,
            num_inference_steps=5,  # Minimum steps needed
            guidance_scale=7.5
        ).images[0]
        from IPython.display import display
        display(image)
    
    # Save and examine the image
    image_path = "debug_image.png"
    image.save(image_path)
    
    # Convert to numpy and check values
    img_array = np.array(image)
    print(f"Image shape: {img_array.shape}")
    print(f"Image dtype: {img_array.dtype}")
    print(f"Image min value: {img_array.min()}")
    print(f"Image max value: {img_array.max()}")
    print(f"Image mean value: {img_array.mean()}")
    print(f"Image has NaN: {np.isnan(img_array).any()}")
    print(f"Image saved to {image_path}")
    
    # Check if image is all black
    if img_array.mean() < 0.01:  # Very dark image check
        print("WARNING: Generated image appears to be all black or very dark!")
        
        # Try to visualize by drastically increasing brightness
        enhanced = np.clip(img_array * 10, 0, 255).astype(np.uint8)
        enhanced_img = Image.fromarray(enhanced)
        enhanced_img.save("enhanced_debug_image.png")
        print("Saved brightness-enhanced version to enhanced_debug_image.png")
    
    # Display alternatives
    print("\nIf you're still having issues, try:")
    print("1. Checking both saved images in your file explorer")
    print("2. Using a different model like: runwayml/stable-diffusion-v1-5")
    print("3. Using a completely different library like PIL to generate a test image")

except Exception as e:
    print(f"Error occurred: {str(e)}")
    print("\nTry a simple PIL test to verify basic image handling:")
    
    # Create a simple test image with PIL
    test_img = Image.new('RGB', (256, 256), color='red')
    test_img.save("pil_test.png")
    print("Created a simple red test image with PIL: pil_test.png")